# HW5
This notebook uses IMF GDP numbers from the Wikipedia page on nominal GDP by country and makes a stacked interactive bar chart


In [24]:
import pandas as pd
import plotly.express as px
import numpy as np
import requests
import plotly.graph_objects as go

In [11]:
# PART 1
# Cleaned Wikipedia data here as a simple Python list

url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

headers = {"User-Agent": "Mozilla/5.0"}
page = requests.get(url, headers=headers)

tables = pd.read_html(page.text)

df = tables[2]
df = df.iloc[:, [0,1]].copy()   # first column = country, second = IMF GDP
df.columns = ["Country", "GDP"]

df["GDP"] = pd.to_numeric(df["GDP"].str.replace(",", "", regex=False), errors="coerce")
df = df.dropna(subset=["GDP"])

df.head()

,Country,GDP
0,World,123584494.0
1,United States,31821293.0
2,China[n 1],20650754.0
3,Germany,5328184.0
4,India,4515629.0


In [17]:
# Create the interactive stacked bar chart

print(df.columns)
df.head()
df = df.rename(columns={
    "Country": "country",
    "GDP": "gdp_musd"
})

region_map = {
    "United States": "North America",
    "Canada": "North America",
    "Mexico": "North America",
    
    "China": "Asia",
    "India": "Asia",
    "Japan": "Asia",
    "South Korea": "Asia",
    
    "Germany": "Europe",
    "United Kingdom": "Europe",
    "France": "Europe",
    "Italy": "Europe",
    
    "Brazil": "South America",
    
    "Australia": "Oceania"
}

df["region"] = df["country"].map(region_map)
df = df.dropna(subset=["region"])

fig = px.bar(
    df,
    x="region",
    y="gdp_musd",
    color="country",
    title="GDP by Country Stacked Within Regions (IMF numbers from Wikipedia)",
    labels={
        "region": "Region",
        "gdp_musd": "GDP (millions of US$)",
        "country": "Country"
    },
    hover_data={"gdp_musd": ":,"}
)

fig.update_layout(barmode="stack")
fig.show()

Index(['country', 'gdp_musd'], dtype='object')


In [18]:
# Save the chart to an html file
fig.write_html("stacked_bar.html")

In [27]:
# PART 2
# Pick one subject
id = 127

# Load hierarchy
url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"
multilevel_lookup = pd.read_csv(url, sep="\t").drop(["Level5"], axis=1)

multilevel_lookup = multilevel_lookup.rename(columns={
    "modify": "roi",
    "modify.1": "level4",
    "modify.2": "level3",
    "modify.3": "level2",
    "modify.4": "level1"
})

multilevel_lookup = multilevel_lookup[["roi", "level4", "level3", "level2", "level1"]]

# Load subject data
subjectData = pd.read_csv(
    "https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv"
)

subjectData = subjectData.loc[
    (subjectData.type == 1) & (subjectData.level == 5) & (subjectData.id == id)
]

subjectData = subjectData[["roi", "volume"]]

# Merge in hierarchy
subjectData = pd.merge(subjectData, multilevel_lookup, on="roi")

# Add intracranial volume label and composition
subjectData = subjectData.assign(icv="ICV")
subjectData = subjectData.assign(comp=subjectData.volume / np.sum(subjectData.volume))

subjectData.head()

,roi,volume,level4,level3,level2,level1,icv,comp
0,SFG_L,12926,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009350
1,SFG_R,10050,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.007270
2,SFG_PFC_L,12783,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009247
3,SFG_PFC_R,11507,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.008324
4,SFG_pole_L,3078,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.002227


In [ ]:
# Keep as many levels as I want
levels = ["icv", "level1", "level2", "level3"]

# Give nodes unique names by level so repeated labels do not collide
for col in levels:
    subjectData[col + "_node"] = col + ": " + subjectData[col].astype(str)

node_cols = [col + "_node" for col in levels]

# Build links between consecutive levels
link_frames = []

for i in range(len(node_cols) - 1):
    temp = subjectData.groupby([node_cols[i], node_cols[i+1]])["comp"].sum().reset_index()
    temp.columns = ["source", "target", "value"]
    link_frames.append(temp)

links = pd.concat(link_frames, ignore_index=True)

# Create node list
nodes = pd.unique(links[["source", "target"]].values.ravel())
node_dict = {name: i for i, name in enumerate(nodes)}

# Map names to numeric ids
links["source_id"] = links["source"].map(node_dict)
links["target_id"] = links["target"].map(node_dict)

# Sankey
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=18,
        label=list(nodes)
    ),
    link=dict(
        source=links["source_id"],
        target=links["target_id"],
        value=links["value"]
    )
)])

fig.update_layout(
    title_text="MRICloud Type 1 hierarchy as a Sankey diagram (Subject 127)",
    font_size=11,
    width=1100,
    height=700
)

fig.show()
fig.write_html("sankey.html")

In [ ]:
# Live Sankey Diagram
# The interactive Sankey diagram can be viewed here: https://laurenc32massie-ship-it.github.io/HW5/